# Verifikasi PLS-SEM dengan seminr (R) - Skripsi Shopee
Tujuan: memeriksa apakah angka Bab 4 (hasil kode Python) sama dengan hasil paket `seminr` yang mapan.

**Sebelum mulai:** pastikan runtime R (Runtime -> Change runtime type -> R). Unggah `smartpls_A_n190.csv` lewat ikon folder di panel kiri (Files -> Upload). Jalankan sel dari atas ke bawah.

## Langkah 1. Pasang dan muat seminr (beberapa menit saat pertama kali)

In [ ]:
install.packages("seminr")
library(seminr)
packageVersion("seminr")

## Langkah 2. Baca data (harus 190 baris, 16 kolom)

In [ ]:
d <- read.csv("smartpls_A_n190.csv")
cat("Dimensi data:", dim(d), "(harus 190 x 16)\n")
stopifnot(nrow(d) == 190)
items <- setdiff(names(d), "ID")
d <- d[, items]
head(d, 3)

## Langkah 3. Definisi model (5 konstruk reflektif, Mode A, 6 jalur hipotesis)

In [ ]:
mm <- constructs(
  composite("EXP",  multi_items("EXP", 1:3),  weights = mode_A),
  composite("CONF", multi_items("CONF", 1:3), weights = mode_A),
  composite("SAT",  multi_items("SAT", 1:3),  weights = mode_A),
  composite("EW",   multi_items("EW", 1:3),   weights = mode_A),
  composite("REP",  multi_items("REP", 1:3),  weights = mode_A)
)

sm <- relationships(
  paths(from = "EXP",  to = c("CONF", "SAT")),   # H1, H2
  paths(from = "CONF", to = c("SAT", "EW")),     # H3, H4
  paths(from = "SAT",  to = "EW"),               # H5
  paths(from = "EW",   to = "REP")               # H6
)

m <- estimate_pls(data = d, measurement_model = mm, structural_model = sm,
                  inner_weights = path_weighting)
s <- summary(m)
cat("Estimasi selesai. Iterasi:", s$iterations, "\n")

## Langkah 4. Hasil mentah seminr (tanpa bootstrap)

In [ ]:
print(s$paths)         # koefisien jalur dan R2
print(s$loadings)     # outer loading
print(s$reliability)  # alpha, rhoC, AVE, rhoA
print(s$validity$htmt)
print(s$validity$fl_criteria)
print(s$fSquare)
print(s$vif_antecedents)

## Langkah 5. Pengecekan otomatis terhadap Bab 4
Kolom `selisih` = seminr dikurangi Bab 4. Selisih kurang dari 0,01 dianggap cocok (pembulatan tiga desimal di Bab 4).

In [ ]:
tol <- 0.01

# Koefisien jalur (Tabel 4.8)
ref_paths <- data.frame(
  hip  = paste0("H", 1:6),
  from = c("EXP","EXP","CONF","CONF","SAT","EW"),
  to   = c("CONF","SAT","SAT","EW","EW","REP"),
  bab4 = c(0.468, 0.199, 0.659, -0.034, 0.716, 0.697))
ref_paths$seminr  <- mapply(function(f, t) m$path_coef[f, t], ref_paths$from, ref_paths$to)
ref_paths$selisih <- round(ref_paths$seminr - ref_paths$bab4, 3)
ref_paths$cocok   <- abs(ref_paths$selisih) < tol
print(ref_paths)

# R2 (Tabel 4.9)
ref_r2 <- data.frame(konstruk = c("CONF","SAT","EW","REP"), bab4 = c(0.219, 0.597, 0.478, 0.485))
ref_r2$seminr  <- sapply(ref_r2$konstruk, function(k) m$rSquared["Rsq", k])
ref_r2$selisih <- round(ref_r2$seminr - ref_r2$bab4, 3)
ref_r2$cocok   <- abs(ref_r2$selisih) < tol
print(ref_r2)

# Outer loading (Tabel 4.4)
ref_ol <- data.frame(
  item = c("EXP1","EXP2","EXP3","CONF1","CONF2","CONF3","SAT1","SAT2","SAT3","EW1","EW2","EW3","REP1","REP2","REP3"),
  bab4 = c(0.904,0.886,0.829,0.859,0.820,0.870,0.908,0.891,0.891,0.862,0.906,0.910,0.893,0.931,0.880))
ref_ol$konstruk <- sub("[0-9]+$", "", ref_ol$item)
ref_ol$seminr   <- mapply(function(i, k) m$outer_loadings[i, k], ref_ol$item, ref_ol$konstruk)
ref_ol$selisih  <- round(ref_ol$seminr - ref_ol$bab4, 3)
ref_ol$cocok    <- abs(ref_ol$selisih) < tol
print(ref_ol)

# Reliabilitas (Tabel 4.5)
rel <- as.data.frame(s$reliability)
ref_rel <- data.frame(
  konstruk = c("EXP","CONF","SAT","EW","REP"),
  alpha_bab4 = c(0.844,0.808,0.878,0.873,0.885),
  rhoC_bab4  = c(0.906,0.886,0.925,0.922,0.928),
  AVE_bab4   = c(0.763,0.722,0.804,0.797,0.812),
  rhoA_bab4  = c(0.846,0.817,0.879,0.876,0.894))
ref_rel$alpha_selisih <- round(rel[ref_rel$konstruk, "alpha"] - ref_rel$alpha_bab4, 3)
ref_rel$rhoC_selisih  <- round(rel[ref_rel$konstruk, "rhoC"]  - ref_rel$rhoC_bab4, 3)
ref_rel$AVE_selisih   <- round(rel[ref_rel$konstruk, "AVE"]   - ref_rel$AVE_bab4, 3)
ref_rel$rhoA_selisih  <- round(rel[ref_rel$konstruk, "rhoA"]  - ref_rel$rhoA_bab4, 3)
print(ref_rel)

semua_cocok <- all(ref_paths$cocok, ref_r2$cocok, ref_ol$cocok)
cat("\nRINGKASAN: jalur, R2, dan loading semuanya cocok dengan Bab 4? ", semua_cocok, "\n")

## Langkah 6. Bootstrapping 5.000 sub-sampel
`seminr` melaporkan CI persentil, sedangkan kode Python memakai BCa, jadi batas CI boleh berbeda tipis. Yang dibandingkan adalah nilai t dan keputusan signifikan (apakah CI memuat 0). Waktu proses beberapa menit.

In [ ]:
set.seed(20261102)
b  <- bootstrap_model(seminr_model = m, nboot = 5000, cores = parallel::detectCores(), seed = 20261102)
sb <- summary(b, alpha = 0.05)
print(sb$bootstrapped_paths)      # bandingkan t dan CI dengan Tabel 4.8
print(sb$bootstrapped_loadings)   # Tabel 4.4
print(sb$bootstrapped_htmt)       # Tabel 4.7 (HTMT dan CI)

## Langkah 7. Efek tidak langsung (Tabel 4.10)

In [ ]:
# Efek tidak langsung spesifik; bandingkan dengan Tabel 4.10
print(specific_effect_significance(b, from = "CONF", through = "SAT", to = "EW",  alpha = 0.05))   # 0,472
print(specific_effect_significance(b, from = "SAT",  through = "EW",  to = "REP", alpha = 0.05))   # 0,499
print(specific_effect_significance(b, from = "EXP",  through = "CONF", to = "SAT", alpha = 0.05))  # 0,308
print(sb$bootstrapped_total_paths)

## Langkah 8. Model alternatif (jalur tambahan SAT -> REP dan CONF -> REP)
Bandingkan dengan Tabel 4.12: EW->REP 0,286; SAT->REP 0,436; CONF->REP 0,214; R2 REP 0,692.

In [ ]:
sm2 <- relationships(
  paths(from = "EXP",  to = c("CONF", "SAT")),
  paths(from = "CONF", to = c("SAT", "EW", "REP")),
  paths(from = "SAT",  to = c("EW", "REP")),
  paths(from = "EW",   to = "REP")
)
m2 <- estimate_pls(data = d, measurement_model = mm, structural_model = sm2,
                   inner_weights = path_weighting)
print(summary(m2)$paths)
b2 <- bootstrap_model(seminr_model = m2, nboot = 5000, cores = parallel::detectCores(), seed = 20261102)
print(summary(b2, alpha = 0.05)$bootstrapped_paths)

## Langkah 9. Simpan hasil
File akan muncul di panel kiri (Files). Klik titik tiga di sebelah file, lalu Download.

In [ ]:
write.csv(ref_paths, "cek_jalur_vs_bab4.csv", row.names = FALSE)
write.csv(ref_ol,    "cek_loading_vs_bab4.csv", row.names = FALSE)
write.csv(sb$bootstrapped_paths, "seminr_bootstrap_jalur.csv")
cat("Tersimpan di panel Files.\n")

## Cara membaca hasil
- **Cocok:** semua kolom `cocok` bernilai TRUE dan keputusan hipotesis (CI memuat 0 atau tidak) sama dengan Tabel 4.8 -> tulis di naskah bahwa hasil diverifikasi dengan seminr (cantumkan versi paket).
- **Tidak cocok:** jangan ubah Bab 4 dulu. Kirim keluaran Langkah 5 ke pembimbing penyusun kode untuk dicari penyebabnya.